# SimpleText Clean Pipeline — Local Qwen3.5 35B-A3B on Kaggle

هذه نسخة منظمة من النوتبوك تعمل على نموذج Qwen المضاف من Kaggle Add Models بدل API خارجي.

الهدف:
- تحميل بيانات SimpleText.
- بناء stratified sample بطريقة ثابتة وعادلة.
- تشغيل نموذج Qwen محليًا من `/kaggle/input`.
- فصل التخطيط عن التنفيذ.
- حفظ النتائج تدريجيًا عبر checkpoints.
- تقييم النتائج على validation/internal test عند توفر المراجع.

> ملاحظة مهمة: `qwen3.5-35b-a3b` نموذج كبير. حتى مع 4-bit quantization قد يحتاج GPU قوي وذاكرة كافية. إذا حصل OOM، خففي `SAMPLE_SIZE` أو استخدمي نموذج أصغر مثل Qwen 7B/14B.


## CELL 1 — Install and Imports

هذه الخلية تجهز المكتبات الأساسية. اجعلي `INSTALL_OR_UPGRADE_PACKAGES = True` فقط إذا احتجت تحديث transformers/accelerate داخل Kaggle.


In [1]:
# ============================================================
# Stable setup for Qwen2.5 AWQ on Kaggle
# Run this cell once, then restart the kernel.
# ============================================================

!pip uninstall -y transformers tokenizers autoawq awq optimum auto-gptq gptqmodel

!pip install -q "transformers==4.51.3"
!pip install -q "tokenizers>=0.21.0,<0.22.0"
!pip install -q "autoawq==0.2.9"
!pip install -q --upgrade accelerate safetensors sentencepiece einops

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 84.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 100.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 7.3 MB/s eta 0:00:00a 0:00:01


In [2]:
!pip install -q evaluate sacremoses sacrebleu textstat
!pip install -q sacremoses textstat sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 11.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 49.4 MB/s eta 0:00:00:00:01


In [ ]:
import os
import signal

print("Restarting kernel now...")
os.kill(os.getpid(), signal.SIGTERM)

In [1]:
import transformers
import tokenizers
import torch
import awq

print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("Torch:", torch.__version__)
print("AWQ:", awq.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/usr/local/lib/python3.12/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
2026-05-01 08:58:45.128505: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:177762

Transformers: 4.51.3
Tokenizers: 0.21.4
Torch: 2.10.0+cu128
AWQ: 0.2.9
CUDA available: True
GPU: Tesla T4


/usr/local/lib/python3.12/dist-packages/torch/jit/_script.py:1480: DeprecationWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [2]:
import transformers
import tokenizers
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Transformers: 4.51.3
Tokenizers: 0.21.4
Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [3]:
INSTALL_OR_UPGRADE_PACKAGES = False

if INSTALL_OR_UPGRADE_PACKAGES:
    import sys
    import subprocess

    packages = [
        "transformers>=4.51.0",
        "accelerate",
        "bitsandbytes",
        "sentencepiece",
        "evaluate",
        "sacrebleu",
        "sacremoses",
        "textstat",
        "tqdm",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])

import os
import re
import gc
import json
import time
import random
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## CELL 2 — Configuration




In [4]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch

In [5]:
# ============================================================
# CELL 2 — Configuration
# ============================================================

from pathlib import Path
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# -------------------------
# Dataset paths
# -------------------------
TRAIN_PATH = Path("/kaggle/input/datasets/raneemrabi3/cochraneauto-sents/cochraneauto_sents_train.csv")
VALID_PATH = Path("/kaggle/input/datasets/raneemrabi3/cochraneauto-sents/cochraneauto_sents_val.csv")
TEST_INTERNAL_PATH = Path("/kaggle/input/datasets/raneemrabi3/cochraneauto-sents/cochraneauto_sents_test.csv")
OFFICIAL_TEST_JSON = Path("/kaggle/input/datasets/raneemrabi3/testjson/simpletext25_task11_test (1).json")
MEDSIMPLIFY_PATH = Path("/kaggle/input/datasets/raneemrabi3/medsimplify/MedSimplify.csv")

# -------------------------
# Output paths
# -------------------------
PROJECT_DIR = Path("/kaggle/working/simpletext_qwen25_14b_clean")
OUTPUT_DIR = PROJECT_DIR / "outputs"
RUN_DIR = OUTPUT_DIR / "runs"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# Local model path
# -------------------------
# Current selected model:
# Qwen2.5 14B Instruct AWQ
MODEL_PATH_OVERRIDE = Path(
    "/kaggle/input/models/qwen-lm/qwen2.5/transformers/14b-instruct-awq/1"
)

MODEL_KEYWORDS = ["qwen", "2.5", "14b", "instruct", "awq"]
REJECT_MODEL_KEYWORDS = ["base", "gptq"]

USE_4BIT = False
TRUST_REMOTE_CODE = True
LOCAL_FILES_ONLY = True

# -------------------------
# Generation settings
# -------------------------
MAX_INPUT_TOKENS = 768
MAX_NEW_TOKENS = 128
PLANNER_MAX_TOKENS = 180

PLANNER_TEMPERATURE = 0.0
EXECUTOR_TEMPERATURE = 0.2

TOP_P = 0.9
GENERATION_MAX_RETRIES = 2
REQUEST_SLEEP_SEC = 0.0
EMPTY_CACHE_EVERY_N_CALLS = 10

# Backward compatibility for any old cell that still uses TEMPERATURE.
# Later, it is better to replace old TEMPERATURE usage with:
# PLANNER_TEMPERATURE or EXECUTOR_TEMPERATURE.
TEMPERATURE = EXECUTOR_TEMPERATURE

# -------------------------
# Sampling settings
# -------------------------
SAMPLE_SIZE = 300
SAMPLE_SPLIT_NAME = "valid_300_stratified_qwen25_14b_awq"
STRATIFY_BY_LABEL_AND_LENGTH = True

# -------------------------
# RAG / terminology support
# -------------------------
RAG_MAX_TERMS = 3

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MODEL_PATH_OVERRIDE:", MODEL_PATH_OVERRIDE)
print("MODEL exists:", MODEL_PATH_OVERRIDE.exists())

print("TRAIN exists:", TRAIN_PATH.exists())
print("VALID exists:", VALID_PATH.exists())
print("TEST_INTERNAL exists:", TEST_INTERNAL_PATH.exists())
print("MEDSIMPLIFY exists:", MEDSIMPLIFY_PATH.exists())

print("MAX_INPUT_TOKENS:", MAX_INPUT_TOKENS)
print("MAX_NEW_TOKENS:", MAX_NEW_TOKENS)
print("PLANNER_MAX_TOKENS:", PLANNER_MAX_TOKENS)
print("PLANNER_TEMPERATURE:", PLANNER_TEMPERATURE)
print("EXECUTOR_TEMPERATURE:", EXECUTOR_TEMPERATURE)

PROJECT_DIR: /kaggle/working/simpletext_qwen25_14b_clean
OUTPUT_DIR: /kaggle/working/simpletext_qwen25_14b_clean/outputs
MODEL_PATH_OVERRIDE: /kaggle/input/models/qwen-lm/qwen2.5/transformers/14b-instruct-awq/1
MODEL exists: True
TRAIN exists: True
VALID exists: True
TEST_INTERNAL exists: True
MEDSIMPLIFY exists: True
MAX_INPUT_TOKENS: 768
MAX_NEW_TOKENS: 128
PLANNER_MAX_TOKENS: 180
PLANNER_TEMPERATURE: 0.0
EXECUTOR_TEMPERATURE: 0.2


## CELL 3 — Basic Text Utilities

In [6]:
def normalize_text(text: object) -> str:
    if pd.isna(text):
        return ""
    text = str(text).replace("\n", " ").replace("\r", " ")
    return re.sub(r"\s+", " ", text).strip()


def normalize_term_text(text: object) -> str:
    text = str(text).strip().lower()
    return re.sub(r"\s+", " ", text)


def safe_word_count(text: object) -> int:
    return len(normalize_text(text).split())


def safe_fkgl(text: object) -> float:
    text = normalize_text(text)
    if not text:
        return np.nan
    try:
        import textstat
        return float(textstat.flesch_kincaid_grade(text))
    except Exception:
        return np.nan


def ensure_row_id(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "row_id" not in df.columns:
        df.insert(0, "row_id", np.arange(len(df)))
    return df


def require_columns(df: pd.DataFrame, columns: List[str], name: str) -> None:
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")

## CELL 4 — Local Qwen Model Setup

هذه الخلية تبحث عن مسار النموذج المضاف من Kaggle، ثم تحمّل tokenizer/model محليًا.


In [7]:
# ============================================================
# CELL 4 — Local Model Path and Loading
# ============================================================

import os
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


def list_kaggle_input_dirs(max_lines: int = 120) -> None:
    printed = 0
    for dirname, _, _ in os.walk("/kaggle/input"):
        print(dirname)
        printed += 1
        if printed >= max_lines:
            print("... output truncated")
            break


def find_local_model_path(
    base_dir="/kaggle/input",
    keywords=None,
    reject_keywords=None,
    override_path=None,
):
    if override_path is not None:
        override_path = Path(override_path)

        if not override_path.exists():
            raise FileNotFoundError(f"MODEL_PATH_OVERRIDE does not exist: {override_path}")

        if not (override_path / "config.json").exists():
            raise FileNotFoundError(f"config.json not found inside: {override_path}")

        return override_path

    keywords = keywords or []
    reject_keywords = reject_keywords or []
    candidates = []

    for root, _, files in os.walk(base_dir):
        root_path = Path(root)
        root_lower = str(root_path).lower()

        if "config.json" not in files:
            continue

        if not all(keyword.lower() in root_lower for keyword in keywords):
            continue

        if any(keyword.lower() in root_lower for keyword in reject_keywords):
            continue

        candidates.append(root_path)

    if not candidates:
        raise FileNotFoundError(
            f"No model path found with keywords={keywords} and reject_keywords={reject_keywords}"
        )

    candidates = sorted(candidates, key=lambda p: len(str(p)))
    return candidates[0]


MODEL_PATH = find_local_model_path(
    keywords=MODEL_KEYWORDS,
    reject_keywords=REJECT_MODEL_KEYWORDS,
    override_path=MODEL_PATH_OVERRIDE,
)

print("Using MODEL_PATH:", MODEL_PATH)
print("config exists:", (MODEL_PATH / "config.json").exists())


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=TRUST_REMOTE_CODE,
    local_files_only=LOCAL_FILES_ONLY,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=TRUST_REMOTE_CODE,
    local_files_only=LOCAL_FILES_ONLY,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

model.eval()

print("Tokenizer loaded:", type(tokenizer).__name__)
print("Model loaded:", type(model).__name__)
print("Device map:", getattr(model, "hf_device_map", "not available"))

Using MODEL_PATH: /kaggle/input/models/qwen-lm/qwen2.5/transformers/14b-instruct-awq/1
config exists: True


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Tokenizer loaded: Qwen2TokenizerFast
Model loaded: Qwen2ForCausalLM
Device map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.layers.32': 1, 'model.layers.33': 1, 'model.layers.34': 1, 'model.layers.35': 1, 'model.layers.36': 1, 'model.layers.37': 1, 'model.layers.38': 1, 'model.layers.39': 1, 'model.layers.40': 1, 'm

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## CELL 5 — Local Chat Generation with Retry

هذه الخلية تستبدل API call بدالة توليد محلية من النموذج.


In [8]:
GENERATION_CALL_COUNT = 0


def first_model_device():
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def build_chat_text(messages: List[dict]) -> str:
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


def clean_model_output(text: str) -> str:
    text = normalize_text(text)

    # Remove common thinking tags if the model emits them.
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)
    text = text.replace("<|im_end|>", "").replace("<|endoftext|>", "")
    text = normalize_text(text)

    return text
# ============================================================
# Local generation call
# ============================================================

import time


def local_chat_call(
    prompt: str,
    system_message: str = "You are a careful text simplification assistant.",
    max_tokens: int = MAX_NEW_TOKENS,
    temperature: float = EXECUTOR_TEMPERATURE,
    retries: int = GENERATION_MAX_RETRIES,
) -> str:
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    ).to(model.device)

    do_sample = temperature is not None and temperature > 0.0

    generation_kwargs = {
        "max_new_tokens": max_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.eos_token_id,
    }

    if do_sample:
        generation_kwargs["temperature"] = temperature
        generation_kwargs["top_p"] = TOP_P

    last_error = None

    for attempt in range(retries):
        try:
            with torch.no_grad():
                output_ids = model.generate(
                    **inputs,
                    **generation_kwargs,
                )

            generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
            decoded = tokenizer.decode(
                generated_ids,
                skip_special_tokens=True,
            ).strip()

            return decoded

        except Exception as exc:
            last_error = exc
            print(f"Generation error on attempt {attempt + 1}/{retries}: {exc}")

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            if REQUEST_SLEEP_SEC > 0:
                time.sleep(REQUEST_SLEEP_SEC)

    raise RuntimeError(f"Generation failed after {retries} retries: {last_error}")


def smoke_test_local_model() -> str:
    prompt = 'Return exactly this JSON: {"ok": true}'
    output = local_chat_call(
        prompt=prompt,
        system_message="Return only valid JSON.",
        max_tokens=30,
        temperature=0.0,
    )
    print("Model response:", output)
    return output


smoke_test_local_model()


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Model response: {"ok": true}


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


'{"ok": true}'

## CELL 6 — Load Data

In [9]:
train_df = ensure_row_id(pd.read_csv(TRAIN_PATH))
valid_df = ensure_row_id(pd.read_csv(VALID_PATH))
test_internal_df = ensure_row_id(pd.read_csv(TEST_INTERNAL_PATH))

require_columns(train_df, ["complex"], "train_df")
require_columns(valid_df, ["complex"], "valid_df")
require_columns(test_internal_df, ["complex"], "test_internal_df")

for df in [train_df, valid_df, test_internal_df]:
    df["complex"] = df["complex"].apply(normalize_text)
    if "simple" in df.columns:
        df["simple"] = df["simple"].apply(normalize_text)
    if "label" in df.columns:
        df["label"] = df["label"].fillna("unknown").astype(str)

print("Train shape:", train_df.shape)
print("Valid shape:", valid_df.shape)
print("Internal test shape:", test_internal_df.shape)

display(train_df.head(3))

Train shape: (11510, 11)
Valid shape: (1697, 11)
Internal test shape: (1512, 11)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,row_id,pair_id,para_id,sent_id,complex,label,simple,simp_sent_id,doc_pos,doc_quint,doc_len
0,0,CD012936,0,0,Three studies (146 participants) met our selec...,rephrase,['Three studies involving 146 participants wer...,0,0.090909,1,11
1,1,CD012936,0,1,"Two studies compared multidisciplinary, fast-t...",delete,[],1,0.181818,1,11
2,2,CD012936,0,2,Two were RCTs with parallel design (total 94 p...,delete,[],1,0.272727,2,11


In [10]:
def token_length(text, tokenizer):
    return len(tokenizer.encode(str(text), add_special_tokens=False))

for name, df in [("train", train_df), ("valid", valid_df)]:
    lengths = df["complex"].apply(lambda x: token_length(x, tokenizer))

    print(f"\n{name.upper()} complex token lengths")
    print("count:", len(lengths))
    print("mean :", int(lengths.mean()))
    print("p50  :", int(lengths.quantile(0.50)))
    print("p90  :", int(lengths.quantile(0.90)))
    print("p95  :", int(lengths.quantile(0.95)))
    print("p99  :", int(lengths.quantile(0.99)))
    print("max  :", int(lengths.max()))


TRAIN complex token lengths
count: 11510
mean : 38
p50  : 28
p90  : 74
p95  : 105
p99  : 188
max  : 557

VALID complex token lengths
count: 1697
mean : 37
p50  : 28
p90  : 73
p95  : 99
p99  : 158
max  : 463


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [11]:
for name, df in [("train", train_df), ("valid", valid_df)]:
    if "simple" in df.columns:
        simple_lengths = df["simple"].apply(lambda x: token_length(x, tokenizer))

        print(f"\n{name.upper()} simple token lengths")
        print("mean :", int(simple_lengths.mean()))
        print("p50  :", int(simple_lengths.quantile(0.50)))
        print("p90  :", int(simple_lengths.quantile(0.90)))
        print("p95  :", int(simple_lengths.quantile(0.95)))
        print("p99  :", int(simple_lengths.quantile(0.99)))
        print("max  :", int(simple_lengths.max()))


TRAIN simple token lengths
mean : 20
p50  : 18
p90  : 46
p95  : 57
p99  : 85
max  : 178

VALID simple token lengths
mean : 19
p50  : 18
p90  : 45
p95  : 54
p99  : 94
max  : 223


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## CELL 7 — MedSimplify Glossary and RAG Term Extraction

In [12]:
med_lookup: Dict[str, str] = {}

if MEDSIMPLIFY_PATH.exists():
    medsimplify_df = pd.read_csv(MEDSIMPLIFY_PATH)
    if not {"word", "definition"}.issubset(set(medsimplify_df.columns)):
        raise ValueError("MedSimplify file must contain columns: word, definition")

    for _, row in medsimplify_df.iterrows():
        term = normalize_term_text(row["word"])
        definition = normalize_text(row["definition"])
        if term and definition:
            med_lookup[term] = definition

print("MedSimplify terms loaded:", len(med_lookup))

RAG_STOP_TERMS = {
    "however", "therefore", "thus", "but", "and", "or", "although",
    "capacity", "measure", "difference", "evidence", "intervention",
    "result", "results", "change", "changes", "risk", "group",
    "groups", "study", "studies", "outcome", "outcomes",
    "evaluate", "effect", "effects", "reduce", "increase",
    "significant", "conclude", "benefit", "oral", "treatment",
    "treatments", "show", "report", "compare", "find", "found",
    "available", "review", "determine", "observe", "detect",
    "mouth", "potential", "moderate", "pregnancy", "regarding",
    "adverse", "complete", "follow-up", "long-term",
    "previous", "individual"
}

WEAK_SINGLE_TERMS = {
    "disease", "therapy", "reaction", "control", "dose", "symptoms",
    "screening", "incidence", "outcome", "outcomes", "pressure",
    "heart", "blood", "pain", "function", "management", "care",
    "chronic", "acute", "cerebral", "pulmonary", "renal",
    "cardiovascular", "medication", "inflammation", "rehabilitation",
    "physician", "biopsy", "fibrillation", "infarction"
}

SAFE_TERM_REPLACEMENTS = {
    "tachycardia": "rapid heart rate",
    "myocardial infarction": "heart attack",
    "atrial fibrillation": "irregular heartbeat",
    "cerebrovascular accident": "stroke",
    "hypotension": "low blood pressure",
    "hypertension": "high blood pressure",
}


def is_useful_term(term: str, definition: str) -> bool:
    term = normalize_term_text(term)
    definition = normalize_text(definition)

    if not term or not definition:
        return False
    if term in RAG_STOP_TERMS:
        return False
    if len(term.split()) == 1 and len(term) <= 3:
        return False
    if len(term.split()) == 1 and term in WEAK_SINGLE_TERMS:
        return False
    if len(definition) < 5:
        return False

    return True


sorted_terms = sorted(med_lookup.keys(), key=len, reverse=True)
multi_word_terms = [t for t in sorted_terms if len(t.split()) >= 2]
single_word_terms = [t for t in sorted_terms if len(t.split()) == 1]


def _match_terms_from_list(
    sentence_lower: str,
    terms: List[str],
    found: Dict[str, str],
    occupied_spans: List[Tuple[int, int]],
    max_terms: int,
) -> None:
    for term in terms:
        if len(found) >= max_terms:
            break

        term = normalize_term_text(term)
        definition = med_lookup.get(term, "")

        if not is_useful_term(term, definition):
            continue

        pattern = r"\b" + re.escape(term) + r"\b"

        for match in re.finditer(pattern, sentence_lower):
            start, end = match.span()
            overlaps = any(not (end <= s or start >= e) for s, e in occupied_spans)

            if overlaps:
                continue

            found[term] = definition
            occupied_spans.append((start, end))
            break


def extract_terms(sentence: str, max_terms: int = RAG_MAX_TERMS) -> Dict[str, str]:
    sentence_lower = normalize_text(sentence).lower()
    found: Dict[str, str] = {}
    occupied_spans: List[Tuple[int, int]] = []

    _match_terms_from_list(sentence_lower, multi_word_terms, found, occupied_spans, max_terms)
    _match_terms_from_list(sentence_lower, single_word_terms, found, occupied_spans, max_terms)

    return found


def build_rag_context(found_terms: Dict[str, str]) -> str:
    if not found_terms:
        return ""

    lines = ["Useful medical term definitions:"]
    for term, definition in found_terms.items():
        replacement = SAFE_TERM_REPLACEMENTS.get(term)
        if replacement:
            lines.append(f"- {term}: {definition} Suggested plain wording: {replacement}.")
        else:
            lines.append(f"- {term}: {definition}")

    return "\n".join(lines)

MedSimplify terms loaded: 3100


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## CELL 8 — Stratified Sample

In [13]:
def add_length_bucket(df: pd.DataFrame, text_col: str = "complex") -> pd.DataFrame:
    df = df.copy()

    def bucket(text: object) -> str:
        n_words = safe_word_count(text)
        if n_words < 12:
            return "short"
        if n_words < 25:
            return "medium"
        return "long"

    df["len_bucket"] = df[text_col].apply(bucket)
    return df


def add_strata_column(
    df: pd.DataFrame,
    text_col: str = "complex",
    label_col: str = "label",
    use_label_and_length: bool = True,
) -> pd.DataFrame:
    df = add_length_bucket(df, text_col=text_col)

    if use_label_and_length and label_col in df.columns:
        df["strata"] = df[label_col].fillna("unknown").astype(str) + "__" + df["len_bucket"]
    elif label_col in df.columns:
        df["strata"] = df[label_col].fillna("unknown").astype(str)
    else:
        df["strata"] = df["len_bucket"]

    return df


def allocate_stratified_counts(counts: pd.Series, n_samples: int) -> Dict[str, int]:
    counts = counts[counts > 0].astype(int)

    if n_samples >= counts.sum():
        return counts.to_dict()

    ideal = counts / counts.sum() * n_samples
    base = np.floor(ideal).astype(int)

    if (base == 0).any():
        base.loc[:] = 0
        largest = counts.sort_values(ascending=False).head(n_samples).index
        base.loc[largest] = 1

    remaining = n_samples - int(base.sum())

    if remaining > 0:
        remainders = (ideal - np.floor(ideal)).sort_values(ascending=False)
        for stratum in remainders.index:
            if remaining <= 0:
                break
            if base[stratum] < counts[stratum]:
                base[stratum] += 1
                remaining -= 1

    elif remaining < 0:
        removable = base.sort_values(ascending=True)
        for stratum in removable.index:
            if remaining == 0:
                break
            if base[stratum] > 0:
                base[stratum] -= 1
                remaining += 1

    return base.astype(int).to_dict()


def stratified_sample(
    df: pd.DataFrame,
    n_samples: int = SAMPLE_SIZE,
    random_state: int = SEED,
    text_col: str = "complex",
    label_col: str = "label",
    use_label_and_length: bool = STRATIFY_BY_LABEL_AND_LENGTH,
) -> pd.DataFrame:
    if n_samples <= 0:
        raise ValueError("n_samples must be positive")

    df = add_strata_column(
        df,
        text_col=text_col,
        label_col=label_col,
        use_label_and_length=use_label_and_length,
    )

    n_samples = min(n_samples, len(df))
    counts = df["strata"].value_counts()
    allocation = allocate_stratified_counts(counts, n_samples)

    sampled_parts = []
    for stratum, k in allocation.items():
        if k <= 0:
            continue

        part = df[df["strata"] == stratum]
        sampled_parts.append(part.sample(n=k, random_state=random_state))

    sampled = pd.concat(sampled_parts, axis=0)
    sampled = sampled.sample(frac=1, random_state=random_state).reset_index(drop=True)

    return sampled


def compare_distribution(full_df: pd.DataFrame, sample_df: pd.DataFrame, col: str) -> pd.DataFrame:
    full_dist = full_df[col].value_counts(normalize=True).rename("full_ratio")
    sample_dist = sample_df[col].value_counts(normalize=True).rename("sample_ratio")
    report = pd.concat([full_dist, sample_dist], axis=1).fillna(0)
    report["diff"] = report["sample_ratio"] - report["full_ratio"]
    return report.sort_values("full_ratio", ascending=False)


valid_subset = stratified_sample(valid_df, n_samples=SAMPLE_SIZE, random_state=SEED)
train_subset = stratified_sample(train_df, n_samples=SAMPLE_SIZE, random_state=SEED)

print("valid_subset shape:", valid_subset.shape)
print("train_subset shape:", train_subset.shape)

if "label" in valid_df.columns:
    display(compare_distribution(valid_df, valid_subset, "label"))

display(valid_subset[["row_id", "complex", "simple", "label", "strata"]].head(5))

valid_subset shape: (300, 13)
train_subset shape: (36, 13)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,full_ratio,sample_ratio,diff
label,,,
rephrase,0.446671,0.446667,-0.000004
delete,0.348851,0.350000,0.001149
ignore,0.098998,0.096667,-0.002332
merge,0.037714,0.040000,0.002286
split,0.034178,0.033333,-0.000845
none,0.033589,0.033333,-0.000255


,row_id,complex,simple,label,strata
0,1226,"Of these, 15 studies with 5787 participants us...","['Of these, 15 studies compared alpha-blockers...",rephrase,rephrase__short
1,1676,There appeared to be a short-term effect of ap...,['There appeared to be a short-term effect of ...,ignore,ignore__long
2,947,CBT is effective in reducing the symptoms of f...,[],delete,delete__long
3,640,"Large, well-conducted RCTs investigating diffe...",['More well-designed and well-reported trials ...,rephrase,rephrase__medium
4,1260,No data were reported relating to healthcare p...,[],delete,delete__short


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## CELL 9 — Planner Prompt

In [14]:
# ============================================================
# Planner
# ============================================================

PLANNER_SYSTEM = """
You are a sentence simplification planner.
Return only strict JSON.
""".strip()


def parse_json_object(text: str) -> dict:
    text = normalize_text(text)
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        text = match.group(0)
    return json.loads(text)


def semantic_planner(sentence: str, next_sentence: str = "") -> Tuple[dict, bool, bool]:
    prompt = f"""
Analyze the sentence and return STRICT JSON only.

Return exactly these keys:
{{
  "is_already_simple": boolean,
  "contains_technical_terms": boolean,
  "is_structurally_complex": boolean,
  "has_secondary_statistical_detail": boolean,
  "requires_abbreviation_expansion": boolean,
  "depends_on_next_sentence": boolean,
  "redundant_given_next_sentence": boolean,
  "reason": "short string"
}}

Definitions:
- is_already_simple: true only if the sentence is already easy for a general reader.
- contains_technical_terms: true if the sentence contains medical or scientific jargon, abbreviations, or difficult domain terms.
- is_structurally_complex: true if the sentence has multiple clauses or would benefit from splitting.
- has_secondary_statistical_detail: true if it mainly contains numerical/statistical detail that may be secondary.
- requires_abbreviation_expansion: true if an abbreviation should be expanded or clarified.
- depends_on_next_sentence: true if this sentence cannot be simplified safely without the next sentence.
- redundant_given_next_sentence: true if the next sentence repeats the same meaning more clearly.

Sentence:
{sentence}

Next sentence:
{next_sentence}
""".strip()

    raw = local_chat_call(
        prompt=prompt,
        system_message=PLANNER_SYSTEM,
        max_tokens=PLANNER_MAX_TOKENS,
        temperature=PLANNER_TEMPERATURE,
    )

    planner_json_ok = True
    used_default = False

    try:
        plan = parse_json_object(raw)

        required_keys = [
            "is_already_simple",
            "contains_technical_terms",
            "is_structurally_complex",
            "has_secondary_statistical_detail",
            "requires_abbreviation_expansion",
            "depends_on_next_sentence",
            "redundant_given_next_sentence",
            "reason",
        ]

        for key in required_keys:
            if key not in plan:
                raise ValueError(f"Missing planner key: {key}")

        for key in required_keys[:-1]:
            if isinstance(plan[key], str):
                plan[key] = plan[key].strip().lower() == "true"
            else:
                plan[key] = bool(plan[key])

        plan["reason"] = str(plan["reason"])

    except Exception as parse_error:
        planner_json_ok = False
        used_default = True
        plan = {
            "is_already_simple": False,
            "contains_technical_terms": False,
            "is_structurally_complex": False,
            "has_secondary_statistical_detail": False,
            "requires_abbreviation_expansion": False,
            "depends_on_next_sentence": False,
            "redundant_given_next_sentence": False,
            "reason": f"default_fallback_parse_error: {parse_error}; raw={raw[:200]}",
        }

    return plan, planner_json_ok, used_default

## CELL 10 — Rule-Based Strategy Decision

In [15]:
def count_commas(sentence: str) -> int:
    return str(sentence).count(",")


def is_long_sentence(sentence: str, threshold: int = 22) -> bool:
    return safe_word_count(sentence) >= threshold


def has_clause_markers(sentence: str) -> bool:
    s = f" {str(sentence).lower()} "
    markers = [
        " although ", " however ", " whereas ", " while ",
        " because ", " therefore ", " which ", " that ",
        " and ", " but ", " or ", " with ", " without ",
        " after ", " before "
    ]
    return any(marker in s for marker in markers)


def has_numeric_detail(sentence: str) -> bool:
    s = str(sentence).lower()

    numeric_patterns = [
        r"\d+",
        r"\d+\.\d+",
        r"\d+%",
        r"\d+\s*to\s*\d+",
        r"\d+\s*-\s*\d+",
        r"\(\s*\d+",
    ]

    statistical_terms = [
        "confidence interval", "ci", "risk ratio", "odds ratio",
        "hazard ratio", "mean difference", "standard deviation",
        "p value", "p-value", "range", "median", "interquartile",
        "iqr", "participants", "records", "studies", "trials",
        "sample size",
    ]

    return any(re.search(pattern, s) for pattern in numeric_patterns) or any(term in s for term in statistical_terms)


def has_secondary_evidence_wording(sentence: str) -> bool:
    s = str(sentence).lower()
    phrases = [
        "low-certainty evidence",
        "very low-certainty evidence",
        "low quality evidence",
        "very low quality evidence",
        "certainty of the evidence",
        "quality of the evidence",
        "insufficient evidence",
        "no clear evidence",
        "did not report",
        "not reported",
        "not possible to determine",
        "no studies reported",
        "secondary outcomes",
        "remaining evidence",
    ]
    return any(phrase in s for phrase in phrases)


def is_mostly_statistical_detail(sentence: str) -> bool:
    s = str(sentence).lower()
    n_words = safe_word_count(s)
    if n_words == 0:
        return False

    number_count = sum(ch.isdigit() for ch in s)
    statistical_keywords = [
        "confidence", "interval", "range", "median",
        "mean", "ratio", "risk", "odds",
        "participants", "studies", "trials",
        "sample", "p-value",
    ]
    keyword_hits = sum(1 for keyword in statistical_keywords if keyword in s)

    return has_numeric_detail(s) and (keyword_hits >= 2 or number_count >= 4 or n_words <= 18)

# ============================================================
# Decision rules helpers
# ============================================================

def has_informative_numeric_context(sentence: str) -> bool:
    """
    Detects numeric/statistical sentences that still contain important
    scientific information and should not be deleted aggressively.
    """
    s = str(sentence).lower()

    informative_terms = [
        "trial", "trials",
        "study", "studies",
        "participants", "patients", "infants", "women", "children",
        "follow-up", "follow up",
        "weeks", "months", "years",
        "assessed", "included", "compared", "reported",
        "outcome", "outcomes",
        "intervention", "interventions",
        "effect", "effects",
        "evidence",
        "sample", "median", "range",
    ]

    return any(term in s for term in informative_terms)


def decide_strategy(plan: dict, verified_terms: Optional[Dict[str, str]] = None, sentence: str = "") -> str:
    verified_terms = verified_terms or {}
    sentence = normalize_text(sentence)

    n_words = safe_word_count(sentence)

    long_sentence = n_words >= 26
    very_long_sentence = n_words >= 36
    comma_heavy = count_commas(sentence) >= 3
    clause_heavy = has_clause_markers(sentence)

    numeric_detail = has_numeric_detail(sentence)
    secondary_evidence = has_secondary_evidence_wording(sentence)
    mostly_stats = is_mostly_statistical_detail(sentence)
    informative_numeric = has_informative_numeric_context(sentence)

    has_terms = bool(verified_terms) or plan.get("contains_technical_terms", False)
    needs_abbrev = plan.get("requires_abbreviation_expansion", False)
    structurally_complex = plan.get("is_structurally_complex", False)

    # 1) Keep clearly simple sentences unchanged.
    if plan.get("is_already_simple", False) and not needs_abbrev:
        return "KEEP_AS_IS"

    # 2) Conservative keep rule.
    # Many "ignore" cases are already acceptable and should not be rewritten.
    if (
        n_words <= 18
        and not needs_abbrev
        and not structurally_complex
        and count_commas(sentence) <= 1
        and not secondary_evidence
        and not has_terms
    ):
        return "KEEP_AS_IS"

    # 3) Avoid deleting informative scientific/statistical sentences.
    # Numeric details are not always secondary in Cochrane-style text.
    if (
        mostly_stats
        and not informative_numeric
        and not has_terms
        and n_words <= 16
    ):
        return "DELETE_OR_OMIT"

    if (
        plan.get("has_secondary_statistical_detail", False)
        and not plan.get("depends_on_next_sentence", False)
        and not informative_numeric
        and not has_terms
        and n_words <= 18
    ):
        return "DELETE_OR_OMIT"

    # 4) Evidence wording usually carries meaning.
    # Rephrase it instead of deleting it.
    if secondary_evidence:
        if n_words <= 10 and not informative_numeric:
            return "DELETE_OR_OMIT"
        return "REPHRASE"

    # 5) Long numeric sentences need simplification, not deletion.
    if numeric_detail and very_long_sentence:
        return "SPLIT_AND_SIMPLIFY"

    # 6) Terminology handling.
    # Avoid standalone SUBSTITUTE unless the sentence is short and the terms are central.
    # In most cases, rephrase is safer than direct term substitution.
    if has_terms:
        if long_sentence or comma_heavy or structurally_complex:
            return "SPLIT_AND_SIMPLIFY"
        return "REPHRASE"

    # 7) Structural complexity.
    # Prefer SPLIT_AND_SIMPLIFY for longer complex sentences.
    # Use REPHRASE for medium complexity to avoid unnecessary splitting.
    if structurally_complex or comma_heavy or (long_sentence and clause_heavy):
        if n_words >= 24:
            return "SPLIT_AND_SIMPLIFY"
        return "REPHRASE"

    return "REPHRASE"

## CELL 11 — Execution Prompts

In [16]:
SHARED_RULES = """
Critical rules:
- Preserve the original meaning exactly.
- Do not add information that is not explicitly supported by the sentence.
- Do not invent causes, effects, or explanations.
- Do not over-explain.
- Keep the output short, natural, and clear.
- Output only the final simplified text.
""".strip()

REPHRASE_PROMPT = """
Task: Rewrite this sentence in plainer English for a general reader.

{rag_context}

{shared_rules}

Sentence:
{sentence}

Simplified:
""".strip()

SPLIT_PROMPT = """
Task: Rewrite this sentence in plainer English for a general reader.
Split it into shorter sentences if needed.

{rag_context}

{shared_rules}

Sentence:
{sentence}

Simplified:
""".strip()

DELETE_OR_OMIT_PROMPT = """
Task: Decide whether this sentence should be omitted because it mainly contains
secondary statistical or non-essential detail.

Rules:
- If the sentence should be omitted, output exactly: OMIT
- Otherwise, rewrite it in simpler English without adding information.
- Output only the final result.

Sentence:
{sentence}

Result:
""".strip()

SUBSTITUTE_PROMPT = """
Task: Rewrite this sentence in plainer English for a general reader.
Focus especially on replacing difficult medical terms with simpler wording
using the provided term definitions.

Critical rules:
- Replace a difficult medical term only when the simpler wording is medically safe and accurate.
- If no safe simpler wording exists, keep the original term.
- Do not guess meanings.
- Do not change the affected organ, body system, or condition type.
- Preserve the original meaning exactly.
- Do not add information not explicitly supported by the sentence.
- Keep the sentence natural and concise.
- Output only the final sentence.

{rag_context}

Sentence:
{sentence}

Simplified:
""".strip()


def call_generation_prompt(prompt: str) -> str:
    return local_chat_call(
        prompt=prompt,
        system_message="You are a careful medical text simplification expert.",
        max_tokens=MAX_NEW_TOKENS,
        temperature=EXECUTOR_TEMPERATURE,
    )


def validate_output(original: str, output: str, strategy: str) -> str:
    original = normalize_text(original)
    output = normalize_text(output)
    strategy = str(strategy).upper()

    if strategy == "KEEP_AS_IS":
        return original

    if strategy == "DELETE_OR_OMIT":
        if output.upper() == "OMIT":
            return ""
        return output if output else original

    return output if output else original


def has_real_rag_terms(rag_context: str) -> bool:
    if not rag_context:
        return False
    return any(line.strip().startswith("- ") and ":" in line for line in str(rag_context).splitlines())


def run_rephrase(sentence: str, rag_context: str = "") -> str:
    prompt = REPHRASE_PROMPT.format(
        sentence=sentence,
        rag_context=rag_context,
        shared_rules=SHARED_RULES,
    )
    return call_generation_prompt(prompt)


def run_split(sentence: str, rag_context: str = "") -> str:
    prompt = SPLIT_PROMPT.format(
        sentence=sentence,
        rag_context=rag_context,
        shared_rules=SHARED_RULES,
    )
    return call_generation_prompt(prompt)


def run_delete_or_omit(sentence: str) -> str:
    prompt = DELETE_OR_OMIT_PROMPT.format(sentence=sentence)
    return call_generation_prompt(prompt)


def run_substitute(sentence: str, rag_context: str = "") -> str:
    prompt = SUBSTITUTE_PROMPT.format(
        sentence=sentence,
        rag_context=rag_context,
    )
    return call_generation_prompt(prompt)


def execute_strategy(sentence: str, strategy: str, rag_context: str = "") -> Tuple[str, str]:
    strategy = str(strategy or "").strip().upper()
    original_sentence = normalize_text(sentence)
    working_sentence = original_sentence

    did_substitute = False

    if has_real_rag_terms(rag_context):
        raw_substitute = run_substitute(working_sentence, rag_context=rag_context)
        clean_substitute = validate_output(working_sentence, raw_substitute, "SUBSTITUTE")

        if clean_substitute and clean_substitute.strip() != working_sentence.strip():
            working_sentence = clean_substitute
            did_substitute = True

    if strategy == "KEEP_AS_IS":
        mode = "keep_after_substitute" if did_substitute else "keep"
        return working_sentence, mode

    if strategy == "DELETE_OR_OMIT":
        raw = run_delete_or_omit(working_sentence)
        final = validate_output(working_sentence, raw, strategy)
        mode = "delete_after_substitute" if did_substitute else "delete_or_omit"
        return final, mode

    if strategy == "SPLIT" or strategy == "SPLIT_AND_SIMPLIFY":
        raw = run_split(working_sentence, rag_context=rag_context)
        final = validate_output(working_sentence, raw, strategy)
        mode = "split_after_substitute" if did_substitute else strategy.lower()
        return final, mode

    if strategy == "SUBSTITUTE":
        mode = "substitute_only" if did_substitute else "substitute_no_change"
        return working_sentence, mode

    raw = run_rephrase(working_sentence, rag_context=rag_context)
    final = validate_output(working_sentence, raw, "REPHRASE")
    mode = "rephrase_after_substitute" if did_substitute else "rephrase"
    return final, mode

## CELL 12 — Checkpoint Helpers

In [17]:
def load_done_ids(csv_path: Path, id_col: str = "row_id") -> Tuple[set, pd.DataFrame]:
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        if id_col in df.columns:
            return set(df[id_col].tolist()), df
        return set(), df
    return set(), pd.DataFrame()


def append_row_to_csv(row_dict: dict, csv_path: Path) -> None:
    row_df = pd.DataFrame([row_dict])
    if csv_path.exists():
        row_df.to_csv(csv_path, mode="a", header=False, index=False)
    else:
        row_df.to_csv(csv_path, index=False)


def prepare_split_dir(split_name: str) -> Path:
    split_dir = RUN_DIR / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    return split_dir

## CELL 13 — Planner Stage

In [18]:
def run_planner_stage(
    df: pd.DataFrame,
    split_name: str,
    text_col: str = "complex",
    reset_first: bool = False,
) -> pd.DataFrame:
    split_dir = prepare_split_dir(split_name)
    planner_csv = split_dir / "planner_stage.csv"

    if reset_first and planner_csv.exists():
        planner_csv.unlink()

    done_ids, _ = load_done_ids(planner_csv)
    total = len(df)

    for i, row in tqdm(df.iterrows(), total=total, desc=f"{split_name} planner"):
        row_id = int(row["row_id"]) if "row_id" in df.columns else int(i)

        if row_id in done_ids:
            continue

        sentence = normalize_text(row[text_col])
        next_sentence = normalize_text(df.iloc[i + 1][text_col]) if i < total - 1 else ""

        try:
            found_terms = extract_terms(sentence)
            plan, planner_json_ok, used_default = semantic_planner(sentence, next_sentence)
            strategy = decide_strategy(plan, found_terms, sentence)

            out_row = {
                "row_id": row_id,
                "complex": sentence,
                "simple": normalize_text(row.get("simple", "")),
                "label": normalize_text(row.get("label", "")),
                "next_sentence": next_sentence,
                "rag_terms": ", ".join(found_terms.keys()) if found_terms else "",
                "rag_terms_count": len(found_terms),
                "plan_json": json.dumps(plan, ensure_ascii=False),
                "planner_json_ok": planner_json_ok,
                "used_default": used_default,
                "strategy": strategy,
                "status": "planned",
            }
            append_row_to_csv(out_row, planner_csv)
            time.sleep(REQUEST_SLEEP_SEC)

        except Exception as error:
            error_row = {
                "row_id": row_id,
                "complex": sentence,
                "simple": normalize_text(row.get("simple", "")),
                "label": normalize_text(row.get("label", "")),
                "next_sentence": next_sentence,
                "rag_terms": "",
                "rag_terms_count": 0,
                "plan_json": "",
                "planner_json_ok": False,
                "used_default": True,
                "strategy": "",
                "status": f"planner_error: {error}",
            }
            append_row_to_csv(error_row, planner_csv)
            print(f"Planner stopped at row_id={row_id}: {error}")
            break

    return pd.read_csv(planner_csv) if planner_csv.exists() else pd.DataFrame()

## CELL 14 — Executor Stage

In [19]:
def terms_from_string(rag_terms: object) -> Dict[str, str]:
    terms: Dict[str, str] = {}
    text = normalize_text(rag_terms)

    if not text:
        return terms

    for term in text.split(","):
        term = normalize_term_text(term)
        if term in med_lookup:
            terms[term] = med_lookup[term]

    return terms


def run_executor_stage(
    split_name: str,
    reset_first: bool = False,
) -> pd.DataFrame:
    split_dir = prepare_split_dir(split_name)
    planner_csv = split_dir / "planner_stage.csv"
    executor_csv = split_dir / "executor_stage.csv"

    if not planner_csv.exists():
        raise FileNotFoundError(f"Planner file not found: {planner_csv}")

    if reset_first and executor_csv.exists():
        executor_csv.unlink()

    planner_df = pd.read_csv(planner_csv)
    planner_df = planner_df[planner_df["status"].astype(str).str.startswith("planned")].copy()

    done_ids, _ = load_done_ids(executor_csv)
    total = len(planner_df)

    for _, row in tqdm(planner_df.iterrows(), total=total, desc=f"{split_name} executor"):
        row_id = int(row["row_id"])

        if row_id in done_ids:
            continue

        sentence = normalize_text(row["complex"])
        strategy = normalize_text(row["strategy"])

        try:
            found_terms = terms_from_string(row.get("rag_terms", ""))
            rag_context = build_rag_context(found_terms)
            output, execution_mode = execute_strategy(sentence, strategy, rag_context)

            out_row = {
                "row_id": row_id,
                "complex": sentence,
                "simple": normalize_text(row.get("simple", "")),
                "label": normalize_text(row.get("label", "")),
                "strategy": strategy,
                "execution_mode": execution_mode,
                "rag_terms": normalize_text(row.get("rag_terms", "")),
                "rag_terms_count": int(row.get("rag_terms_count", 0)),
                "planner_json_ok": bool(row.get("planner_json_ok", False)),
                "used_default": bool(row.get("used_default", False)),
                "output": normalize_text(output),
                "output_changed": normalize_text(output).lower() != sentence.lower(),
                "status": "executed",
            }
            append_row_to_csv(out_row, executor_csv)
            time.sleep(REQUEST_SLEEP_SEC)

        except Exception as error:
            error_row = {
                "row_id": row_id,
                "complex": sentence,
                "simple": normalize_text(row.get("simple", "")),
                "label": normalize_text(row.get("label", "")),
                "strategy": strategy,
                "execution_mode": "",
                "rag_terms": normalize_text(row.get("rag_terms", "")),
                "rag_terms_count": int(row.get("rag_terms_count", 0)),
                "planner_json_ok": bool(row.get("planner_json_ok", False)),
                "used_default": bool(row.get("used_default", False)),
                "output": "",
                "output_changed": False,
                "status": f"executor_error: {error}",
            }
            append_row_to_csv(error_row, executor_csv)
            print(f"Executor stopped at row_id={row_id}: {error}")
            break

    return pd.read_csv(executor_csv) if executor_csv.exists() else pd.DataFrame()

## CELL 15 — Run One Small Test First

شغلي هذه الخلية قبل العينة الكبيرة. إذا ظهر OOM، خففي النموذج أو حجم العينة.


In [20]:
debug_subset = valid_subset.head(5).copy()
debug_split_name = "debug_5_qwen35_a3b"

debug_planner_df = run_planner_stage(
    debug_subset,
    split_name=debug_split_name,
    reset_first=True,
)

debug_executor_df = run_executor_stage(
    split_name=debug_split_name,
    reset_first=True,
)

display(debug_executor_df[["complex", "simple", "strategy", "execution_mode", "output"]])


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


debug_5_qwen35_a3b planner:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.d

debug_5_qwen35_a3b executor:   0%|          | 0/5 [00:00<?, ?it/s]

,complex,simple,strategy,execution_mode,output
0,"Of these, 15 studies with 5787 participants us...","['Of these, 15 studies compared alpha-blockers...",DELETE_OR_OMIT,delete_after_substitute,NaN
1,There appeared to be a short-term effect of ap...,['There appeared to be a short-term effect of ...,SPLIT_AND_SIMPLIFY,split_after_substitute,There was about a 50% hair loss within six mon...
2,CBT is effective in reducing the symptoms of f...,[],SPLIT_AND_SIMPLIFY,split_after_substitute,CBT works well in reducing fatigue after treat...
3,"Large, well-conducted RCTs investigating diffe...",['More well-designed and well-reported trials ...,SPLIT_AND_SIMPLIFY,split_after_substitute,"Large, well-conducted studies comparing differ..."
4,No data were reported relating to healthcare p...,[],SPLIT_AND_SIMPLIFY,split_after_substitute,No information was provided about any harm or ...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## CELL 16 — Run Stratified Sample

In [21]:
planner_df = run_planner_stage(
    valid_subset,
    split_name=SAMPLE_SPLIT_NAME,
    reset_first=True,
)

executor_df = run_executor_stage(
    split_name=SAMPLE_SPLIT_NAME,
    reset_first=True,
)

print("Planner rows:", len(planner_df))
print("Executor rows:", len(executor_df))

display(executor_df.head(5))

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


valid_300_stratified_qwen25_14b_awq planner:   0%|          | 0/300 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.d

valid_300_stratified_qwen25_14b_awq executor:   0%|          | 0/300 [00:00<?, ?it/s]

Planner rows: 300
Executor rows: 300


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,row_id,complex,simple,label,strategy,execution_mode,rag_terms,rag_terms_count,planner_json_ok,used_default,output,output_changed,status
0,1226,"Of these, 15 studies with 5787 participants us...","['Of these, 15 studies compared alpha-blockers...",rephrase,DELETE_OR_OMIT,delete_after_substitute,placebo,1,True,False,NaN,True,executed
1,1676,There appeared to be a short-term effect of ap...,['There appeared to be a short-term effect of ...,ignore,SPLIT_AND_SIMPLIFY,split_after_substitute,"approximately, whereas, hair",3,True,False,There was about a 50% reduction in hair using ...,True,executed
2,947,CBT is effective in reducing the symptoms of f...,[],delete,SPLIT_AND_SIMPLIFY,split_after_substitute,"effective, fatigue, post",3,True,False,CBT works well in reducing fatigue after treat...,True,executed
3,640,"Large, well-conducted RCTs investigating diffe...",['More well-designed and well-reported trials ...,rephrase,SPLIT_AND_SIMPLIFY,split_after_substitute,clinical,1,True,False,"Large, well-conducted studies comparing differ...",True,executed
4,1260,No data were reported relating to healthcare p...,[],delete,SPLIT_AND_SIMPLIFY,split_after_substitute,relating to,1,True,False,No information was provided about any harm or ...,True,executed


## CELL 17 — Evaluation Helpers

In [21]:
def try_load_sari_metric():
    try:
        import evaluate
        return evaluate.load("sari")
    except Exception as error:
        print("Could not load evaluate/sari:", error)
        return None


sari_metric = try_load_sari_metric()


def safe_bleu(prediction: str, reference: str) -> float:
    prediction = normalize_text(prediction)
    reference = normalize_text(reference)

    if not prediction or not reference:
        return np.nan

    try:
        import sacrebleu
        return sacrebleu.sentence_bleu(prediction, [reference]).score / 100.0
    except Exception:
        return np.nan


def safe_sari_batch(sources, predictions, references) -> float:
    if sari_metric is None:
        return np.nan

    try:
        return float(
            sari_metric.compute(
                sources=list(sources),
                predictions=list(predictions),
                references=[[ref] for ref in references],
            )["sari"]
        )
    except Exception as error:
        print("SARI error:", error)
        return np.nan


def safe_sari_sentence(source: str, prediction: str, reference: str) -> float:
    return safe_sari_batch([source], [prediction], [reference])


def add_eval_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for col in ["complex", "simple", "output"]:
        if col not in df.columns:
            df[col] = ""
        df[col] = df[col].fillna("").astype(str).apply(normalize_text)

    if "rag_terms_count" not in df.columns:
        df["rag_terms_count"] = 0
    df["rag_terms_count"] = df["rag_terms_count"].fillna(0).astype(int)

    df["fkgl_src"] = df["complex"].apply(safe_fkgl)
    df["fkgl_pred"] = df["output"].apply(safe_fkgl)
    df["fkgl_delta"] = df["fkgl_pred"] - df["fkgl_src"]
    df["compression"] = df.apply(
        lambda row: safe_word_count(row["output"]) / max(1, safe_word_count(row["complex"])),
        axis=1,
    )
    df["bleu_sentence"] = df.apply(
        lambda row: safe_bleu(row["output"], row["simple"]),
        axis=1,
    )
    df["sari_sentence"] = df.apply(
        lambda row: safe_sari_sentence(row["complex"], row["output"], row["simple"]),
        axis=1,
    )
    df["has_rag_terms"] = df["rag_terms_count"] > 0
    df["output_changed"] = df["complex"].str.lower() != df["output"].str.lower()

    return df


def print_split_report(df: pd.DataFrame, title: str) -> pd.DataFrame:
    report_df = add_eval_columns(df)
    overall_sari = safe_sari_batch(report_df["complex"], report_df["output"], report_df["simple"])

    print("=" * 80)
    print(f"{title} STRATEGY DISTRIBUTION")
    print(report_df["strategy"].value_counts(dropna=False))

    print("\n" + "=" * 80)
    print(f"{title} EXECUTION MODE DISTRIBUTION")
    print(report_df["execution_mode"].value_counts(dropna=False))

    print("\n" + "=" * 80)
    print(f"{title} EVALUATION REPORT")
    print("=" * 80)
    print(f"Total sentences    : {len(report_df)}")
    print(f"Overall SARI       : {overall_sari:.4f}" if not pd.isna(overall_sari) else "Overall SARI       : nan")
    print(f"Avg BLEU           : {np.nanmean(report_df['bleu_sentence']):.4f}")
    print(f"Avg FKGL source    : {np.nanmean(report_df['fkgl_src']):.2f}")
    print(f"Avg FKGL pred      : {np.nanmean(report_df['fkgl_pred']):.2f}")
    print(f"Avg FKGL delta     : {np.nanmean(report_df['fkgl_delta']):.2f}")
    print(f"Avg compression    : {np.nanmean(report_df['compression']):.3f}")
    print(f"Planner JSON OK    : {int(report_df['planner_json_ok'].sum())} / {len(report_df)}")
    print(f"Used default plan  : {int(report_df['used_default'].sum())} / {len(report_df)}")
    print(f"Output changed     : {int(report_df['output_changed'].sum())} / {len(report_df)}")
    print(f"Sentences with RAG : {int(report_df['has_rag_terms'].sum())} / {len(report_df)}")

    print("\nPer-strategy:")
    for strategy in report_df["strategy"].dropna().unique():
        sub = report_df[report_df["strategy"] == strategy]
        strategy_sari = safe_sari_batch(sub["complex"], sub["output"], sub["simple"])
        if pd.isna(strategy_sari):
            print(f"- {strategy}: n={len(sub)}, SARI=nan")
        else:
            print(f"- {strategy}: n={len(sub)}, SARI={strategy_sari:.4f}")

    return report_df

## CELL 18 — Evaluate Completed Run

In [10]:
executor_path = RUN_DIR / SAMPLE_SPLIT_NAME / "executor_stage.csv"

if not executor_path.exists():
    raise FileNotFoundError(f"Executor output not found: {executor_path}")

completed_df = pd.read_csv(executor_path)
report_df = print_split_report(completed_df, SAMPLE_SPLIT_NAME)

report_path = RUN_DIR / SAMPLE_SPLIT_NAME / "evaluation_report.csv"
report_df.to_csv(report_path, index=False)

print("Saved report:", report_path)

display(report_df.sort_values("sari_sentence", ascending=True).head(10)[
    ["complex", "simple", "output", "strategy", "execution_mode", "rag_terms", "sari_sentence"]
])

display(report_df.sort_values("sari_sentence", ascending=False).head(10)[
    ["complex", "simple", "output", "strategy", "execution_mode", "rag_terms", "sari_sentence"]
])

NameError: name 'RUN_DIR' is not defined

In [22]:
import pandas as pd
import numpy as np

PLANNER_CSV = "/kaggle/input/datasets/raneemrabi3/evaluation1/planner_stage (4).csv"
EXECUTOR_CSV = "/kaggle/input/datasets/raneemrabi3/evaluation1/executor_stage.csv"
EVAL_CSV = "/kaggle/input/datasets/raneemrabi3/evaluation1/evaluation_report.csv"

planner_df = pd.read_csv(PLANNER_CSV)
executor_df = pd.read_csv(EXECUTOR_CSV)

print("planner_df:", planner_df.shape)
print("executor_df:", executor_df.shape)
print(executor_df.columns.tolist())

display(executor_df.head(3))

planner_df: (300, 12)
executor_df: (300, 13)
['row_id', 'complex', 'simple', 'label', 'strategy', 'execution_mode', 'rag_terms', 'rag_terms_count', 'planner_json_ok', 'used_default', 'output', 'output_changed', 'status']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,row_id,complex,simple,label,strategy,execution_mode,rag_terms,rag_terms_count,planner_json_ok,used_default,output,output_changed,status
0,1226,"Of these, 15 studies with 5787 participants us...","['Of these, 15 studies compared alpha-blockers...",rephrase,DELETE_OR_OMIT,delete_after_substitute,placebo,1,True,False,NaN,True,executed
1,1676,There appeared to be a short-term effect of ap...,['There appeared to be a short-term effect of ...,ignore,SPLIT_AND_SIMPLIFY,split_after_substitute,"approximately, whereas, hair",3,True,False,There was about a 50% reduction in hair using ...,True,executed
2,947,CBT is effective in reducing the symptoms of f...,[],delete,SPLIT_AND_SIMPLIFY,split_after_substitute,"effective, fatigue, post",3,True,False,CBT works well in reducing fatigue after treat...,True,executed


In [23]:
import evaluate
from sacrebleu.metrics import BLEU
import textstat
import pandas as pd
import numpy as np

df_eval = executor_df.copy()

def clean_eval_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

df_eval["complex"] = df_eval["complex"].apply(clean_eval_text)
df_eval["simple"] = df_eval["simple"].apply(clean_eval_text)
df_eval["output"] = df_eval["output"].apply(clean_eval_text)

sari_metric = evaluate.load("sari")
bleu_metric = BLEU(effective_order=True)

def safe_sentence_sari(source, prediction, reference):
    source = clean_eval_text(source)
    prediction = clean_eval_text(prediction)
    reference = clean_eval_text(reference)

    if source == "" or reference == "":
        return np.nan

    try:
        result = sari_metric.compute(
            sources=[source],
            predictions=[prediction],
            references=[[reference]],
        )
        return float(result["sari"])
    except Exception as e:
        print("SARI error:", type(e).__name__, e)
        return np.nan

def safe_sentence_bleu(prediction, reference):
    prediction = clean_eval_text(prediction)
    reference = clean_eval_text(reference)

    if prediction == "" or reference == "":
        return 0.0

    try:
        return float(bleu_metric.sentence_score(prediction, [reference]).score)
    except Exception:
        return np.nan

def safe_fkgl(text):
    text = clean_eval_text(text)

    if text == "":
        return np.nan

    try:
        return float(textstat.flesch_kincaid_grade(text))
    except Exception:
        return np.nan

def safe_compression(source, prediction):
    source_words = clean_eval_text(source).split()
    pred_words = clean_eval_text(prediction).split()

    if len(source_words) == 0:
        return np.nan

    return len(pred_words) / len(source_words)

df_eval["sari_sentence"] = [
    safe_sentence_sari(src, pred, ref)
    for src, pred, ref in zip(df_eval["complex"], df_eval["output"], df_eval["simple"])
]

df_eval["bleu_sentence"] = [
    safe_sentence_bleu(pred, ref)
    for pred, ref in zip(df_eval["output"], df_eval["simple"])
]

df_eval["fkgl_src"] = df_eval["complex"].apply(safe_fkgl)
df_eval["fkgl_pred"] = df_eval["output"].apply(safe_fkgl)
df_eval["fkgl_delta"] = df_eval["fkgl_pred"] - df_eval["fkgl_src"]

df_eval["compression"] = [
    safe_compression(src, pred)
    for src, pred in zip(df_eval["complex"], df_eval["output"])
]

report_df = df_eval.copy()

print("=" * 80)
print("EVALUATION REPORT")
print("=" * 80)
print("Total sentences    :", len(report_df))
print("Overall SARI       :", np.nanmean(report_df["sari_sentence"]))
print("Avg BLEU           :", np.nanmean(report_df["bleu_sentence"]))
print("Avg FKGL source    :", np.nanmean(report_df["fkgl_src"]))
print("Avg FKGL pred      :", np.nanmean(report_df["fkgl_pred"]))
print("Avg FKGL delta     :", np.nanmean(report_df["fkgl_delta"]))
print("Avg compression    :", np.nanmean(report_df["compression"]))

if "planner_json_ok" in report_df.columns:
    print("Planner JSON OK    :", int(report_df["planner_json_ok"].sum()), "/", len(report_df))

if "used_default" in report_df.columns:
    print("Used default plan  :", int(report_df["used_default"].sum()), "/", len(report_df))

if "output_changed" in report_df.columns:
    print("Output changed     :", int(report_df["output_changed"].sum()), "/", len(report_df))

if "rag_terms_count" in report_df.columns:
    print("Sentences with RAG :", int((report_df["rag_terms_count"] > 0).sum()), "/", len(report_df))

print("\nNaN counts:")
print(report_df[[
    "sari_sentence",
    "bleu_sentence",
    "fkgl_src",
    "fkgl_pred",
    "fkgl_delta",
    "compression"
]].isna().sum())

print("\nPer-strategy:")
for strategy, part in report_df.groupby("strategy"):
    print(
        f"- {strategy}: "
        f"n={len(part)}, "
        f"SARI={np.nanmean(part['sari_sentence']):.3f}, "
        f"BLEU={np.nanmean(part['bleu_sentence']):.3f}, "
        f"Compression={np.nanmean(part['compression']):.3f}")


EVALUATION REPORT
Total sentences    : 300
Overall SARI       : 37.14639467293691
Avg BLEU           : 5.6875041085078495
Avg FKGL source    : 12.883556441639024
Avg FKGL pred      : 10.25000009514424
Avg FKGL delta     : -3.0324077275233345
Avg compression    : 0.8499387165062324
Planner JSON OK    : 300 / 300
Used default plan  : 0 / 300
Output changed     : 292 / 300
Sentences with RAG : 173 / 300

NaN counts:
sari_sentence     0
bleu_sentence     0
fkgl_src          0
fkgl_pred        31
fkgl_delta       31
compression       0
dtype: int64

Per-strategy:
- DELETE_OR_OMIT: n=39, SARI=42.687, BLEU=3.962, Compression=0.166
- KEEP_AS_IS: n=8, SARI=46.790, BLEU=6.449, Compression=1.000
- REPHRASE: n=67, SARI=35.735, BLEU=6.191, Compression=0.941
- SPLIT: n=36, SARI=33.377, BLEU=8.485, Compression=1.002
- SPLIT_AND_SIMPLIFY: n=142, SARI=37.004, BLEU=4.869, Compression=0.937
- SUBSTITUTE: n=8, SARI=31.795, BLEU=11.066, Compression=1.046


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [24]:
cols = ["row_id", "label", "strategy", "execution_mode", "complex", "simple", "output", "sari_sentence", "bleu_sentence", "compression"]

display(
    report_df.sort_values("sari_sentence", ascending=True)[cols].head(20)
)

,row_id,label,strategy,execution_mode,complex,simple,output,sari_sentence,bleu_sentence,compression
262,1410,ignore,DELETE_OR_OMIT,delete_or_omit,Six studies compared individual education to u...,['Six studies compared individual education to...,,1.991422,0.000000,0.000000
214,1205,ignore,REPHRASE,rephrase_after_substitute,Five trials did not report on harmful (adverse...,['Five trials did not report on harmful (adver...,Five studies found no harmful effects. Four st...,5.572480,3.860508,0.820513
160,643,rephrase,DELETE_OR_OMIT,delete_or_omit,Follow-up ranged from six weeks to two years.,['Trials lasted from six weeks to two years.'],,9.093915,0.000000,0.000000
186,511,ignore,DELETE_OR_OMIT,delete_or_omit,This suggests the need for more high-quality s...,['This suggests the need for more high-quality...,This suggests we need more good studies.,10.383598,8.423556,0.875000
104,356,ignore,SPLIT,split,Another reason was that the findings were base...,['Another reason was that the findings were ba...,Another reason is that the findings came from ...,10.449020,11.947337,0.925926
67,373,ignore,REPHRASE,rephrase,There was no dose-related effect of fluvastati...,['There was no dose-related effect of fluvasta...,Fluvastatin did not affect HDL cholesterol lev...,10.506554,6.272848,1.000000
175,1636,ignore,SPLIT_AND_SIMPLIFY,split_after_substitute,But healthcare workers pointed to a lack of tr...,['But healthcare workers pointed to a lack of ...,But healthcare workers said they lacked traini...,11.184641,12.085512,0.842105
52,1574,ignore,SPLIT,split,"These interventions were complex, with more th...",['The telerehabilitation interventions evaluat...,"These interventions were complex, including mu...",11.275497,11.882920,0.950000
167,930,ignore,SUBSTITUTE,substitute_only,Hepatic vascular occlusion does not decrease t...,['Hepatic vascular occlusion does not decrease...,Liver blood vessel blockage does not reduce th...,11.504630,7.193387,1.200000
155,361,ignore,SPLIT,split_after_substitute,Some healthcare workers thought it was importa...,['Some healthcare workers thought it was impor...,Some healthcare workers believe it's important...,11.589044,13.622964,1.000000


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [25]:
display(
    report_df[report_df["strategy"].isin(["SUBSTITUTE", "SPLIT"])]
    .sort_values("sari_sentence", ascending=True)[cols]
    .head(20)
)

,row_id,label,strategy,execution_mode,complex,simple,output,sari_sentence,bleu_sentence,compression
104,356,ignore,SPLIT,split,Another reason was that the findings were base...,['Another reason was that the findings were ba...,Another reason is that the findings came from ...,10.449020,11.947337,0.925926
52,1574,ignore,SPLIT,split,"These interventions were complex, with more th...",['The telerehabilitation interventions evaluat...,"These interventions were complex, including mu...",11.275497,11.882920,0.950000
167,930,ignore,SUBSTITUTE,substitute_only,Hepatic vascular occlusion does not decrease t...,['Hepatic vascular occlusion does not decrease...,Liver blood vessel blockage does not reduce th...,11.504630,7.193387,1.200000
155,361,ignore,SPLIT,split_after_substitute,Some healthcare workers thought it was importa...,['Some healthcare workers thought it was impor...,Some healthcare workers believe it's important...,11.589044,13.622964,1.000000
113,1406,ignore,SPLIT,split,"None of these studies reported night sweats, s...","['None of the studies reported night sweats, s...","None of these studies reported night sweats, p...",13.686816,20.336829,1.000000
111,607,ignore,SPLIT,split,"Each study compared phonics training alone, or...","['Each study compared phonics training alone, ...",Each study compared phonics training alone or ...,14.287903,13.006815,1.000000
194,426,ignore,SPLIT,split,About one-third of the studies were in urban/p...,['About one-third of the studies were in urban...,About one-third of the studies took place in u...,15.330918,22.665851,1.304348
145,663,ignore,SPLIT,split,"However, evidence is of low and very low quali...","['However, evidence is of low and very low qua...","However, the evidence is of low quality.",16.296296,16.947942,0.700000
87,1012,rephrase,SPLIT,split,Four studies with a total of 141 participants ...,['We identified four studies including 141 par...,The review included four studies with a total ...,26.403764,5.243652,1.187500
78,900,rephrase,SPLIT,split,"In addition, we identified two ongoing studies...","['Moreover, we identified two trials, of which...",We found two ongoing studies. One was stopped ...,27.872379,1.819687,0.925926


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/recomputed_eval")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

report_path = OUTPUT_DIR / "evaluation_report_recomputed.csv"
report_df.to_csv(report_path, index=False)

print("Saved:", report_path)

In [26]:
# ============================================================
# Find available DataFrames in notebook
# ============================================================

import pandas as pd

dataframes = []

for name, obj in globals().items():
    if isinstance(obj, pd.DataFrame):
        dataframes.append((name, obj.shape, list(obj.columns)))

for name, shape, cols in dataframes:
    print("\nDATAFRAME:", name)
    print("Shape:", shape)
    print("Columns:", cols[:20])


DATAFRAME: train_df
Shape: (11510, 11)
Columns: ['row_id', 'pair_id', 'para_id', 'sent_id', 'complex', 'label', 'simple', 'simp_sent_id', 'doc_pos', 'doc_quint', 'doc_len']

DATAFRAME: valid_df
Shape: (1697, 11)
Columns: ['row_id', 'pair_id', 'para_id', 'sent_id', 'complex', 'label', 'simple', 'simp_sent_id', 'doc_pos', 'doc_quint', 'doc_len']

DATAFRAME: test_internal_df
Shape: (1512, 11)
Columns: ['row_id', 'pair_id', 'para_id', 'sent_id', 'complex', 'label', 'simple', 'simp_sent_id', 'doc_pos', 'doc_quint', 'doc_len']

DATAFRAME: df
Shape: (1697, 11)
Columns: ['row_id', 'pair_id', 'para_id', 'sent_id', 'complex', 'label', 'simple', 'simp_sent_id', 'doc_pos', 'doc_quint', 'doc_len']

DATAFRAME: medsimplify_df
Shape: (3123, 2)
Columns: ['word', 'definition']

DATAFRAME: valid_subset
Shape: (300, 13)
Columns: ['row_id', 'pair_id', 'para_id', 'sent_id', 'complex', 'label', 'simple', 'simp_sent_id', 'doc_pos', 'doc_quint', 'doc_len', 'len_bucket', 'strata']

DATAFRAME: train_subset
Shap

In [27]:
# ============================================================
# Debug evaluation values from report_df
# ============================================================

df_eval = report_df.copy()

print("Rows:", len(df_eval))
print("Status counts:")
print(df_eval["status"].value_counts(dropna=False))

print("\nNull counts:")
print(df_eval[[
    "complex", "simple", "output",
    "fkgl_src", "fkgl_pred", "fkgl_delta",
    "compression", "bleu_sentence", "sari_sentence"
]].isna().sum())

print("\nEmpty text counts:")
for col in ["complex", "simple", "output"]:
    print(col, (df_eval[col].fillna("").astype(str).str.strip() == "").sum())

print("\nMetric summary:")
display(df_eval[[
    "fkgl_src", "fkgl_pred", "fkgl_delta",
    "compression", "bleu_sentence", "sari_sentence"
]].describe())

display(df_eval[[
    "row_id", "complex", "simple", "output",
    "strategy", "execution_mode",
    "bleu_sentence", "sari_sentence",
    "fkgl_src", "fkgl_pred"
]].head(10))

Rows: 300
Status counts:
status
executed    300
Name: count, dtype: int64

Null counts:
complex            0
simple             0
output             0
fkgl_src         300
fkgl_pred        300
fkgl_delta       300
compression        0
bleu_sentence    300
sari_sentence    300
dtype: int64

Empty text counts:
complex 0
simple 0
output 31

Metric summary:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,fkgl_src,fkgl_pred,fkgl_delta,compression,bleu_sentence,sari_sentence
count,0.0,0.0,0.0,300.000000,0.0,0.0
mean,NaN,NaN,NaN,0.849939,NaN,NaN
std,NaN,NaN,NaN,0.382083,NaN,NaN
min,NaN,NaN,NaN,0.000000,NaN,NaN
25%,NaN,NaN,NaN,0.726010,NaN,NaN
50%,NaN,NaN,NaN,0.925926,NaN,NaN
75%,NaN,NaN,NaN,1.034615,NaN,NaN
max,NaN,NaN,NaN,2.571429,NaN,NaN


,row_id,complex,simple,output,strategy,execution_mode,bleu_sentence,sari_sentence,fkgl_src,fkgl_pred
0,1226,"Of these, 15 studies with 5787 participants us...","['Of these, 15 studies compared alpha-blockers...",,DELETE_OR_OMIT,delete_after_substitute,NaN,NaN,NaN,NaN
1,1676,There appeared to be a short-term effect of ap...,['There appeared to be a short-term effect of ...,There was about a 50% reduction in hair using ...,SPLIT_AND_SIMPLIFY,split_after_substitute,NaN,NaN,NaN,NaN
2,947,CBT is effective in reducing the symptoms of f...,[],CBT works well in reducing fatigue after treat...,SPLIT_AND_SIMPLIFY,split_after_substitute,NaN,NaN,NaN,NaN
3,640,"Large, well-conducted RCTs investigating diffe...",['More well-designed and well-reported trials ...,"Large, well-conducted studies comparing differ...",SPLIT_AND_SIMPLIFY,split_after_substitute,NaN,NaN,NaN,NaN
4,1260,No data were reported relating to healthcare p...,[],No information was provided about any harm or ...,SPLIT_AND_SIMPLIFY,split_after_substitute,NaN,NaN,NaN,NaN
5,1285,No studies reported on live birth rates.,[],No studies reported on live birth rates.,KEEP_AS_IS,keep,NaN,NaN,NaN,NaN
6,537,Five people need to be treated to prevent one ...,[],Five people need to be treated to prevent one ...,REPHRASE,rephrase_after_substitute,NaN,NaN,NaN,NaN
7,1271,"Considering the quality of evidence, we are un...",['We are uncertain whether there is a differen...,We are unsure if there is a difference in mult...,SPLIT_AND_SIMPLIFY,split_and_simplify,NaN,NaN,NaN,NaN
8,195,Fifteen trials (3492 infants) assessed the eff...,['Fourteen trials (3492 infants) assessed the ...,,DELETE_OR_OMIT,delete_after_substitute,NaN,NaN,NaN,NaN
9,100,There was no sufficient clinical similarity be...,[],There was not enough similarity between the tr...,SUBSTITUTE,substitute_only,NaN,NaN,NaN,NaN


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [28]:
# ============================================================
# Manual metric aggregation
# ============================================================

import numpy as np

print("Overall SARI:", np.nanmean(report_df["sari_sentence"]))
print("Avg BLEU:", np.nanmean(report_df["bleu_sentence"]))
print("Avg FKGL source:", np.nanmean(report_df["fkgl_src"]))
print("Avg FKGL pred:", np.nanmean(report_df["fkgl_pred"]))
print("Avg FKGL delta:", np.nanmean(report_df["fkgl_delta"]))
print("Avg compression:", np.nanmean(report_df["compression"]))

Overall SARI: nan
Avg BLEU: nan
Avg FKGL source: nan
Avg FKGL pred: nan
Avg FKGL delta: nan
Avg compression: 0.8499387165062324


/tmp/ipykernel_136/400879387.py:7: RuntimeWarning: Mean of empty slice
  print("Overall SARI:", np.nanmean(report_df["sari_sentence"]))
/tmp/ipykernel_136/400879387.py:8: RuntimeWarning: Mean of empty slice
  print("Avg BLEU:", np.nanmean(report_df["bleu_sentence"]))
/tmp/ipykernel_136/400879387.py:9: RuntimeWarning: Mean of empty slice
  print("Avg FKGL source:", np.nanmean(report_df["fkgl_src"]))
/tmp/ipykernel_136/400879387.py:10: RuntimeWarning: Mean of empty slice
  print("Avg FKGL pred:", np.nanmean(report_df["fkgl_pred"]))
/tmp/ipykernel_136/400879387.py:11: RuntimeWarning: Mean of empty slice
  print("Avg FKGL delta:", np.nanmean(report_df["fkgl_delta"]))


In [29]:
!pip install -q sacremoses textstat sacrebleu

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/usr/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=136) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 18.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 46.1 MB/s eta 0:00:00:00:01


In [2]:
!pip install -q evaluate sacremoses sacrebleu textstat
!pip install -q sacremoses textstat sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00


In [ ]:
import os
import signal

print("Restarting kernel now...")
os.kill(os.getpid(), signal.SIGTERM)

In [11]:
import evaluate
import sacrebleu
import textstat

print("evaluate installed successfully")

evaluate installed successfully


In [12]:
import evaluate

sari_metric = evaluate.load("sari")

test_row = executor_df.iloc[0]

result = sari_metric.compute(
    sources=[str(test_row["complex"])],
    predictions=[str(test_row["output"])],
    references=[[str(test_row["simple"])]],
)

print(result)

NameError: name 'executor_df' is not defined

In [3]:
import evaluate

sari_metric = evaluate.load("sari")

test_row = executor_df.iloc[0]

source = str(test_row["complex"])
prediction = str(test_row["output"])
reference = str(test_row["simple"])

print("SOURCE:", source)
print("PRED:", prediction)
print("REF:", reference)

result = sari_metric.compute(
    sources=[source],
    predictions=[prediction],
    references=[[reference]],
)

print(result)

2026-04-30 17:25:55.500887: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777569955.526758     224 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777569955.535221     224 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777569955.557352     224 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777569955.557376     224 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777569955.557379     224 computation_placer.cc:177] computation placer alr

NameError: name 'executor_df' is not defined

## CELL 19 — Optional: Run on Internal Test

In [ ]:
INTERNAL_TEST_SAMPLE_SIZE = 300
internal_test_split_name = "internal_test_300_stratified_qwen35_a3b"

internal_test_subset = stratified_sample(
    test_internal_df,
    n_samples=INTERNAL_TEST_SAMPLE_SIZE,
    random_state=SEED,
)

internal_planner_df = run_planner_stage(
    internal_test_subset,
    split_name=internal_test_split_name,
    reset_first=True,
)

internal_executor_df = run_executor_stage(
    split_name=internal_test_split_name,
    reset_first=True,
)

internal_report_df = print_split_report(internal_executor_df, internal_test_split_name)


## CELL 20 — Optional: Prepare Official Submission JSON

In [ ]:
def save_submission_json(
    predictions_df: pd.DataFrame,
    output_json_path: Path,
    run_id: str = "SPU_DeepSimplify_task11_hf_llama33_stratified_v1",
) -> None:
    required = ["pair_id", "para_id", "sent_id", "output"]
    missing = [col for col in required if col not in predictions_df.columns]

    if missing:
        raise ValueError(f"Cannot create submission. Missing columns: {missing}")

    rows = []
    for _, row in predictions_df.iterrows():
        rows.append({
            "pair_id": row["pair_id"],
            "para_id": row["para_id"],
            "sent_id": row["sent_id"],
            "output": normalize_text(row["output"]),
        })

    payload = {
        "run_id": run_id,
        "predictions": rows,
    }

    output_json_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_json_path, "w", encoding="utf-8") as file:
        json.dump(payload, file, ensure_ascii=False, indent=2)

    print("Saved submission JSON:", output_json_path)


# Example:
# save_submission_json(official_predictions_df, OUTPUT_DIR / "SPU_DeepSimplify_task11_hf_llama33_stratified_v1.json")